# Feature selection inside the folds

A screening study measured 2,000 features on each of 60 samples, half of them labeled 1. With
far more features than samples, a common first step is to keep only the features most
correlated with the label and train a small classifier on those. Here you write that step
and a 5-fold cross-validation around it, with NumPy alone, and run both on two data sets:
one whose labels carry real signal and one whose labels have nothing to do with the features.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
n, p = 60, 2000

# Labels that have nothing to do with the features.
X_noise = rng.normal(size=(n, p))
y_noise = rng.permutation(np.repeat([0, 1], n // 2))

# Labels that shift the first 10 features up or down.
X_signal = rng.normal(size=(n, p))
y_signal = rng.permutation(np.repeat([0, 1], n // 2))
X_signal[:, :10] += 0.6 * (2 * y_signal[:, None] - 1)

# Five folds of 12 rows, fixed for the whole notebook.
folds = np.array_split(np.random.default_rng(0).permutation(n), 5)


def nearest_centroid(X_train, y_train, X_val):
    """Predict 1 for the rows of X_val closer to the mean of the class-1 training rows."""
    c0 = X_train[y_train == 0].mean(axis=0)
    c1 = X_train[y_train == 1].mean(axis=0)
    d0 = ((X_val - c0) ** 2).sum(axis=1)
    d1 = ((X_val - c1) ** 2).sum(axis=1)
    return (d1 < d0).astype(int)

## Part 1 (coding task): The most correlated features

Write `top_features(X, y, m)`, which returns the indices of the `m` columns of `X` whose
Pearson correlation with `y` is largest in absolute value, the largest first, as an integer
array of length `m`.

The correlation of column $j$ with $y$ is

$$
r_j = \frac{\sum_i (x_{ij} - \bar{x}_j)(y_i - \bar{y})}{\sqrt{\sum_i (x_{ij} - \bar{x}_j)^2 \, \sum_i (y_i - \bar{y})^2}} .
$$

Compute all 2,000 of them at once, without a loop over the columns.

In [ ]:
def top_features(X, y, m):
    """Indices of the m columns most correlated with y in absolute value, largest first."""
    # TODO
    raise NotImplementedError

In [ ]:
small_y = np.array([0, 0, 0, 1, 1, 1])
small_X = np.array([[0, 3, 100, 1],
                    [1, 3, 160, 0],
                    [0, 2, 130, 1],
                    [1, 0, 150, 0],
                    [2, 0, 140, 1],
                    [1, 1, 170, 0]], dtype=float)
got = np.asarray(top_features(small_X, small_y, 2))
assert got.shape == (2,), f"expected 2 indices, got shape {got.shape}"
assert np.issubdtype(got.dtype, np.integer), "return column indices as integers"
assert 1 in got, "column 1 falls as y rises (correlation -0.93): rank by the absolute value of the correlation"
assert 2 not in got, ("column 2 has the largest covariance with y only because its values are about 100 times larger: "
                      "divide by the spread of each column, which is what makes it a correlation")
assert list(got) == [1, 0], f"expected [1, 0] (correlations -0.93 and 0.73), got {list(got)}"
ref = np.corrcoef(X_noise[:, :300].T, y_noise)[-1, :-1]
got = np.asarray(top_features(X_noise[:, :300], y_noise, 15))
assert list(got) == list(np.argsort(-np.abs(ref))[:15]), "on the noise data the ranking differs from the absolute correlations"
print("Part 1 passed")

## Part 2 (coding task): Cross-validating the whole procedure

Write `cv_accuracy(X, y, folds, m)`. For each fold in `folds` (a list of index arrays), it treats
that fold as the validation rows and the other folds as the training rows, ranks the `m` features
with `top_features` on the training rows only, fits `nearest_centroid` on the training rows
restricted to those features, and records the accuracy on the validation rows. It returns the mean
of the five accuracies.

The question to answer with your function: what accuracy should someone expect from this procedure
on new samples?


In [ ]:
def cv_accuracy(X, y, folds, m):
    """Mean validation accuracy over the folds of: keep m features, then nearest centroid."""
    # TODO
    raise NotImplementedError

In [ ]:
noise_acc = cv_accuracy(X_noise, y_noise, folds, 20)
signal_acc = cv_accuracy(X_signal, y_signal, folds, 10)
print(f"pure noise, 20 features: {noise_acc:.3f}")
print(f"real signal, 10 features: {signal_acc:.3f}")
assert noise_acc < 0.7, (
    f"on labels unrelated to the features your estimate is {noise_acc:.3f}, yet no procedure can beat a coin "
    "on new samples here: the validation rows took part in a step of the fit")
assert signal_acc > 0.85, (
    f"on the data with real signal the estimate is only {signal_acc:.3f}: choose the features from each "
    "fold's training rows, not its validation rows, and fit the centroids on those rows")
assert abs(noise_acc - 0.4) < 0.4 and abs(signal_acc - 0.9166666666666666) < 0.15, (
    "close, but not the expected values (about 0.400 and 0.917 on this NumPy release; the exact values "
    "depend on the seeded Generator, whose stream NumPy does not guarantee across versions): choose the "
    "features and fit the centroids on each fold's training rows only, score its validation rows, and use "
    "every fold once")
print("Part 2 passed")


## Solution

### Part 1: The most correlated features

In [ ]:
def top_features(X, y, m):
    """Indices of the m columns most correlated with y in absolute value, largest first."""
    Xc = X - X.mean(axis=0)
    yc = y - y.mean()
    r = (Xc * yc[:, None]).sum(axis=0) / np.sqrt((Xc ** 2).sum(axis=0) * (yc ** 2).sum())
    return np.argsort(-np.abs(r))[:m]

**Why it works.** Centering each column and the label turns the numerator into a dot product, and
`(Xc * yc[:, None]).sum(axis=0)` takes all 2,000 dot products in one pass. Dividing by both spreads
makes the score independent of each feature's units, so column 2, which has the largest covariance
with y only because of its scale, drops out of the top two. Sorting by $-|r|$ keeps strongly
negative features, which predict the label just as well as positive ones.


### Part 2: Cross-validating the whole procedure

In [ ]:
def cv_accuracy(X, y, folds, m):
    """Mean validation accuracy over the folds of: keep m features, then nearest centroid."""
    scores = []
    for i, val in enumerate(folds):
        train = np.concatenate([f for j, f in enumerate(folds) if j != i])
        keep = top_features(X[train], y[train], m)          # chosen from the training rows only
        pred = nearest_centroid(X[train][:, keep], y[train], X[val][:, keep])
        scores.append(np.mean(pred == y[val]))
    return float(np.mean(scores))

**Why it works.** The feature ranking is fit to data just as the centroids are, so it is part of the
model and must be redone on each fold's training rows. Ranking once on all 60 rows lets
every validation row help choose the 20 features that happen to separate its own label
best among 2,000 random ones, and the noise data then score about 0.87 in cross-validation
while new samples would score 0.5. Done inside the folds, the noise data score 0.40, a coin
flip within the spread of five folds of 12, and the signal data still score 0.92, so the
honest procedure loses nothing where there is something to find.